# Rótulos adjudicados — CheXpert agora, VinDr depois

A primeira linha de base mediu **AUROC macro 0,664** no NIH ChestX-ray14, cujos
rótulos são minerados por NLP dos laudos. Fibrose saiu em **0,448**, com IC95
[0,421, 0,473] — excluindo 0,5. Isso é ambíguo: o modelo é ruim em fibrose, ou
"Fibrosis" no PadChest e no NIH não nomeiam o mesmo achado?

**Só rótulo adjudicado por radiologista resolve.** Este notebook cobre os dois caminhos:

| | Parte 1 — CheXpert | Parte 2 — VinDr-CXR |
|---|---|---|
| Rótulos | 3 radiologistas, voto majoritário | 5 radiologistas |
| Acesso | registro Stanford AIMI | credenciamento PhysioNet |
| Prazo | imediato | 24–48 h após o curso CITI (~6 h) |
| Tamanho | 234 imagens (só frontais entram) | 3.000 exames |
| Automatizável | **sim, aqui** | **não** — ver Parte 2 |

### O CheXpert não responde a pergunta sobre fibrose

Registro isso antes de você rodar, porque é o tipo de detalhe que se perde no
entusiasmo de um número novo: das 14 observações do CheXpert, **nenhuma é fibrose**.
A interseção com a nomenclatura do torchxrayvision cobre cardiomegalia, edema,
consolidação, atelectasia, derrame e mais alguns — e é sobre esses que a Parte 1
troca rótulo de NLP por rótulo de consenso.

Fibrose, enfisema, nódulo, massa, infiltrado, espessamento pleural e hérnia ficam
**sem contrapartida** e serão listados em `not_evaluated[]` com essa justificativa.
Quem tem fibrose anotada por radiologista é o VinDr-CXR — ou seja, o achado que
levantou a pergunta é exatamente o que exige a Parte 2. A Parte 1 é valiosa mesmo
assim: se o AUROC subir nos achados compartilhados, o rótulo do NIH é parte do
problema em geral, e isso muda a leitura do 0,448 sem medi-lo diretamente.

---

## O que este notebook não faz

A Parte 2 **não** faz o cadastro por você. O curso CITI são 6 h de módulos com a sua
assinatura, e o formulário da PhysioNet exige identidade verificável e um referenciador
que eles contactam. Uma credencial obtida por terceiro é revogada quando descoberta.

O que a Parte 2 faz: gera os textos prontos e **valida o seu relatório CITI antes do
envio** — três erros comuns custam uma rodada de revisão e são detectáveis em segundos.


## 0. Ambiente


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

BRANCH = 'claude/roadmap-interpretacao-radiologica-vh15ek'
REPO = 'https://github.com/drguilhermecapel/radiologyai.git'
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()


def eh_clone(p):
    py = p / 'pyproject.toml'
    return py.is_file() and 'name = "radiologyai"' in py.read_text(encoding='utf-8')


aqui = Path.cwd()
REPO_DIR = next((c for c in [aqui, *aqui.parents] if eh_clone(c)), None) or BASE / 'radiologyai'
if not eh_clone(REPO_DIR):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(REPO_DIR)], check=True)
else:
    # Nunca descarta trabalho não commitado.
    sujo = subprocess.run(['git','-C',str(REPO_DIR),'status','--porcelain'],
                          capture_output=True, text=True, check=True).stdout.strip()
    if sujo:
        print('Há mudanças locais — não vou atualizar o clone.')
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','--depth','1','origin',BRANCH], check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','FETCH_HEAD'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')

import importlib.util
falta = [m for m in ('pydicom','pydantic','typer','torchxrayvision','PIL','yaml','pypdf')
         if importlib.util.find_spec(m) is None]
if falta:
    subprocess.run([sys.executable,'-m','pip','install','-q','pydicom==2.4.4','pydantic>=2.7',
                    'pydantic-settings>=2.3','typer>=0.12','torchxrayvision','scikit-image',
                    'pillow','PyYAML','pypdf'], check=True)

_spec = importlib.util.find_spec('radiologyai')
if not (_spec and _spec.origin and str(REPO_DIR) in _spec.origin):
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    '--ignore-requires-python','-e','.'], check=True)
for m in [m for m in list(sys.modules) if m == 'radiologyai' or m.startswith('radiologyai.')]:
    del sys.modules[m]

import radiologyai
_log = subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True)
print('código:', _log.stdout.strip())
print('radiologyai', radiologyai.__version__,
      '| python', radiologyai.selftest()['python'])


---
# PARTE 1 — CheXpert

## 1.1 Obter os dados

**Antes de rodar:** acesse [aimi.stanford.edu/datasets/chexpert-chest-x-rays](https://aimi.stanford.edu/datasets/chexpert-chest-x-rays),
preencha o formulário e aceite o *Stanford University Research Use Agreement*. O
portal entrega um link individual (normalmente Azure, via `azcopy`).

Baixe o **`CheXpert-v1.0-small`** (~11 GB): a resolução reduzida basta, porque o
modelo redimensiona para 224×224 de qualquer forma.

A célula abaixo aceita **quatro formas de entrada** — preencha só uma:

| Variável | Quando usar |
|---|---|
| `CAMINHO_LOCAL` | já baixou, no Colab ou no Drive |
| `COMANDO_AZCOPY` | cole o comando `azcopy copy ...` que o portal deu |
| `URL_DIRETA` | um link http(s) direto para o zip |
| nenhuma | faz upload manual do `valid.csv` + imagens |

**Só o conjunto de validação é usado** (234 imagens). Se você já tem o `valid.csv` e a
pasta `valid/`, não precisa do dataset inteiro.


In [ ]:
CAMINHO_LOCAL = None      # ex.: '/content/drive/MyDrive/CheXpert-v1.0-small'
COMANDO_AZCOPY = None     # ex.: 'azcopy copy "https://..." . --recursive'
URL_DIRETA = None         # ex.: 'https://.../CheXpert-v1.0-small.zip'

CHEX = Path(os.environ.get('RADIOLOGYAI_CHEXPERT_ROOT', str(BASE / 'chexpert')))
CHEX.mkdir(parents=True, exist_ok=True)


def valid_csv_presente(raiz: Path):
    for c in (raiz / 'valid.csv', raiz / 'CheXpert-v1.0-small' / 'valid.csv',
              raiz / 'CheXpert-v1.0' / 'valid.csv'):
        if c.is_file():
            return c.parent
    achados = [p.parent for p in raiz.rglob('valid.csv')]
    return achados[0] if achados else None


if CAMINHO_LOCAL:
    CHEX = Path(CAMINHO_LOCAL)
elif COMANDO_AZCOPY:
    if not shutil.which('azcopy'):
            subprocess.run(
            'curl -sL https://aka.ms/downloadazcopy-v10-linux'
            ' | tar -xz --strip-components=1 -C /usr/local/bin --wildcards "*/azcopy"',
            shell=True, check=True)
    subprocess.run(COMANDO_AZCOPY, shell=True, cwd=CHEX, check=True)
elif URL_DIRETA:
    destino = CHEX / 'chexpert.zip'
    subprocess.run(['curl','-L','-o',str(destino),URL_DIRETA], check=True)
    shutil.unpack_archive(str(destino), str(CHEX))
    destino.unlink()
elif valid_csv_presente(CHEX) is None:
    print('Nenhuma fonte configurada. Faça upload do valid.csv e da pasta valid/,')
    print('ou preencha uma das variáveis acima e rode de novo.')
    try:
        from google.colab import files
        enviados = files.upload()
        for nome in enviados:
            alvo = CHEX / nome
            shutil.move(nome, alvo)
            if alvo.suffix in ('.zip', '.gz', '.tar'):
                shutil.unpack_archive(str(alvo), str(CHEX))
    except ImportError:
        pass

RAIZ = valid_csv_presente(CHEX)
if RAIZ is None:
    raise RuntimeError(f'valid.csv não encontrado em {CHEX}. Veja as opções na célula acima.')
print('CheXpert em:', RAIZ)


## 1.2 Conferir antes de medir

Falha aqui é barata; descobrir depois da inferência que o conjunto estava incompleto,
não. O carregador já trata três casos que mudam o número:

- **rótulo incerto (`-1.0`) vira ausente**, nunca negativo — contá-lo como negativo
  inflaria a especificidade artificialmente;
- **laterais são filtradas**, porque o uso pretendido declarado cobre só frontais;
- **nenhum achado é colapsado** em outro: o que não tem contrapartida é declarado.


In [ ]:
from radiologyai.data.chexpert import build_valid_manifest

manifest = build_valid_manifest(RAIZ, frontal_only=True)
print(f'{len(manifest)} imagens frontais · {len(manifest.patient_ids)} pacientes')
print('sha256 do manifest:', manifest.sha256()[:16], '…')
print()
for nome in manifest.label_names:
    n = manifest.positives(nome)
    marca = '  ' if n >= 10 else '! '   # abaixo de 10 não há suporte para AUROC
    print(f'{marca}{nome:<28} {n:>4} positivos ({n/len(manifest):.1%})')
print()
print('Achados com ! terão suporte insuficiente e serão declarados em not_evaluated[].')


## 1.3 Medir contra o consenso

234 imagens rodam em segundos, mesmo em CPU. O artefato registra `git_sha`,
`weights_sha256`, `manifest_sha256`, seed e versões — reprodutível por terceiro.


In [ ]:
import torch

from radiologyai.evaluation.baseline import run_baseline, summarize

ARTIFACTS = Path(os.environ.get('RADIOLOGYAI_ARTIFACTS_DIR', str(BASE / 'artifacts' / 'eval')))
if (BASE / 'drive' / 'MyDrive').is_dir() and not os.environ.get('RADIOLOGYAI_ARTIFACTS_DIR'):
    ARTIFACTS = BASE / 'drive' / 'MyDrive' / 'radiologyai' / 'artifacts' / 'eval'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', DEVICE, '| artefatos:', ARTIFACTS)

resultado = run_baseline(
    RAIZ, ARTIFACTS,
    dataset='chexpert-valid',
    card_id='xrv-densenet121-pc',
    manifest_path=REPO_DIR / 'datasets' / 'manifests' / 'chexpert_valid.csv',
    device=DEVICE, batch_size=32, n_bootstrap=2000, seed=20260101,
)
print()
summarize(resultado)


## 1.4 A comparação que importa

Mesmo modelo, mesmos pesos. Muda só a **origem do rótulo**: NLP versus consenso de
radiologistas. Onde o AUROC subir muito, o problema era o rótulo do NIH, não o modelo.

A tabela só tem linha para os achados que os dois datasets nomeiam. Os que aparecem
apenas no NIH — fibrose entre eles — continuam sem resposta até a Parte 2.


In [ ]:
import json

nih = None
for p in sorted((REPO_DIR / 'artifacts' / 'eval').glob('*/metrics.json')):
    m = json.loads(p.read_text())
    if m['dataset']['name'].startswith('NIH'):
        nih = m   # o mais recente
chex = resultado.metrics

if nih is None:
    print('Nenhum artefato do NIH neste checkout — rode o notebook 01 antes para comparar.')
else:
    print(f"AUROC macro   NIH(NLP) {nih['macro_auroc']:.3f}"
          f"   CheXpert(3 radiologistas) {chex['macro_auroc']:.3f}")
    print()
    print(f"{'achado':<22}{'NIH':>8}{'CheXpert':>10}{'delta':>9}   {'IC95 CheXpert':<18}")
    for nome in sorted(chex['per_label'], key=lambda n: -chex['per_label'][n]['auroc']):
        b = chex['per_label'][nome]
        a = nih['per_label'].get(nome)
        lo, hi = b['auroc_ci95']
        if a:
            print(f"{nome:<22}{a['auroc']:>8.3f}{b['auroc']:>10.3f}"
                  f"{b['auroc']-a['auroc']:>+9.3f}   [{lo:.3f},{hi:.3f}]")
        else:
            print(f"{nome:<22}{'—':>8}{b['auroc']:>10.3f}{'—':>9}"
                  f"   [{lo:.3f},{hi:.3f}]")
    print()
    print('Um delta grande e positivo indica rótulo do NIH ruidoso para aquele achado.')
    print(f'ATENÇÃO: {len(manifest)} imagens dão IC largo — um delta dentro do IC'
          ' não é conclusivo.')
    ausentes = [n for n in nih['per_label'] if n not in chex['per_label']]
    if ausentes:
        print('Sem contrapartida no CheXpert, logo sem resposta aqui:',
              ', '.join(sorted(ausentes)))


In [ ]:
# Zip do artefato, para commitar no repositório
zip_path = shutil.make_archive(
    str(resultado.run_dir.parent / 'chexpert_baseline'), 'zip', resultado.run_dir)
print('zip:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Pegue o zip pelo painel de arquivos, à esquerda.')


---
# PARTE 2 — Credenciamento PhysioNet (VinDr-CXR)

**Isto não é automatizável.** O curso CITI são ~6 h de módulos com a sua assinatura; o
formulário exige identidade verificável e um referenciador que a PhysioNet contacta
diretamente. Fazer por você seria falsidade ideológica, e a credencial cairia.

O que segue: os textos prontos e um verificador que evita a rodada de revisão perdida.


## 2.1 Kit de inscrição

Gera um arquivo com tudo que o formulário pede, já preenchido.


In [ ]:
from datetime import date

DADOS = {
    'Nome completo': 'Guilherme Capel Pasqua',
    'Profissão': 'Physician (médico)',
    'Registro profissional': 'CRM-SP 175873 · RQE 137036',
    'País': 'Brazil',
    'Instituição': '<preencha: vinculação atual — InCor/USP se o doutorado servir>',
    'Referenciador': '<nome, cargo, instituição e e-mail institucional>',
}

DESCRICAO = (
    'Retrospective evaluation of deep learning models for chest radiograph '
    'interpretation. I am a cardiologist (CRM-SP 175873) and PhD candidate in '
    'Cardiology. The project benchmarks published chest X-ray models against '
    'radiologist-adjudicated labels to quantify the gap between NLP-mined and '
    'expert-adjudicated ground truth, with per-finding confidence intervals and '
    'subgroup analysis. No clinical deployment is intended; the work is '
    'research-only and results are published with full provenance. Data will not '
    'be redistributed and will remain on encrypted storage under my control.'
)

PASSOS = [
    '1. CITI (~6 h, o gargalo) — about.citiprogram.org → Register',
    '   Afiliação: "Massachusetts Institute of Technology Affiliates"',
    '   Curso: "Data or Specimens Only Research" (NÃO o completo de sujeitos humanos)',
    '   Ao fim: baixe o COMPLETION REPORT (não o Certificate) em Records → View-Print-Share',
    '2. Conta — physionet.org/register/ (e-mail institucional acelera)',
    '3. Credenciamento — physionet.org/settings/credentialing/',
    '4. Revisão: 24–48 h após o envio do Report',
    '5. VinDr-CXR — physionet.org/content/vindr-cxr/ → aceitar a licença → baixar',
]

linhas = ['KIT DE CREDENCIAMENTO PHYSIONET', f'gerado em {date.today().isoformat()}', '',
          'DADOS DO FORMULÁRIO', '']
linhas += [f'  {k}: {v}' for k, v in DADOS.items()]
linhas += ['', 'DESCRIÇÃO DA PESQUISA (copie e cole)', '', DESCRICAO, '', 'PASSOS', '', *PASSOS]
texto = '\n'.join(linhas)

kit = Path('physionet_kit.txt')
kit.write_text(texto, encoding='utf-8')
print(texto)
print()
print('gravado em', kit.resolve())


## 2.2 Validar o relatório CITI antes de enviar

Três erros custam uma rodada de revisão — dias de espera — e levam segundos para detectar:

1. enviar o **Completion Certificate** em vez do **Completion Report** (o CITI oferece
   os dois lado a lado, e só o Report serve);
2. ter feito o curso errado;
3. relatório vencido.

Rode esta célula quando tiver o PDF e faça o upload.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('citi', REPO_DIR / 'scripts' / 'check_citi_report.py')
citi = importlib.util.module_from_spec(spec)
sys.modules['citi'] = citi
spec.loader.exec_module(citi)

RELATORIO = None   # ex.: '/content/citi_report.pdf'

if RELATORIO is None:
    try:
        from google.colab import files
        print('Faça upload do PDF do relatório CITI:')
        enviados = files.upload()
        RELATORIO = next(iter(enviados), None)
    except ImportError:
        pass

if RELATORIO:
    print(f'\nVerificando {RELATORIO}\n')
    citi.verificar(RELATORIO, nome=DADOS['Nome completo']).relatar()
else:
    print('Nenhum relatório enviado. Volte aqui quando concluir o curso CITI.')


---

## Depois

**Parte 1:** me mande o `chexpert_baseline.zip`. Eu integro o artefato, gero o
relatório e atualizo o model card — o segundo dataset entra no mesmo registro de
proveniência.

**Parte 2:** com a credencial aprovada, o VinDr-CXR entra como um terceiro adaptador
de dataset. 3.000 exames contra 234 estreita os intervalos de confiança o bastante
para conclusões por achado — que é o que a Fase 3 (modelo próprio) precisa para ser
medida contra algo que valha a pena.

**O que nenhuma das duas partes entrega:** validação clínica. Estes são desempenhos
isolados de algoritmo, retrospectivos. Ver [`HONEST_STATUS.md`](../HONEST_STATUS.md).
